In [0]:
%sql
CREATE SCHEMA de_workspace26.ecommerce_schema_michael;
CREATE VOLUME de_workspace26.ecommerce_schema_michael.ecommerce_volume_michael;


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS de_workspace26.michaelBronze;
CREATE SCHEMA IF NOT EXISTS de_workspace26.michaelSilver;
CREATE SCHEMA IF NOT EXISTS de_workspace26.michaelGold;
CREATE VOLUME IF NOT EXISTS de_workspace26.michaelBronze.bronze;
CREATE VOLUME IF NOT EXISTS de_workspace26.michaelSilver.silver;
CREATE VOLUME IF NOT EXISTS de_workspace26.michaelGold.gold;

In [0]:
base_path = "/Volumes/de_workspace26/ecommerce_schema_michael/ecommerce_volume_michael/raw/"

orders_df = spark.read.csv(base_path + "orders.csv", header=True, inferSchema=True)

customers_df = spark.read.csv(base_path + "customers.csv", header=True, inferSchema=True)

products_df = spark.read.csv(base_path + "products.csv", header=True, inferSchema=True)

In [0]:
orders_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.michaelBronze.orders")

customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.michaelBronze.customers")

products_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.michaelBronze.products")

In [0]:
%sql
ALTER TABLE de_workspace26.michaelBronze.orders
ADD CONSTRAINT order_id_not_null CHECK (order_id IS NOT NULL);

In [0]:
%sql
DESCRIBE HISTORY de_workspace26.michaelBronze.orders;

In [0]:
%sql
USE CATALOG de_workspace26;
USE SCHEMA ecommerce_schema_michael;

In [0]:
dbutils.fs.mkdirs('/Volumes/de_workspace26/ecommerce_schema_michael/ecommerce_volume_michael/raw')

In [0]:
display(dbutils.fs.ls("/Volumes/de_workspace26/ecommerce_schema_michael/ecommerce_volume_michael/raw/"))

In [0]:
orders_stream = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/de_workspace26/michaelBronze/bronze/schema/orders_stream") \
    .load(base_path)

In [0]:
orders_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/de_workspace26/michaelBronze/bronze/checkpoints/orders_stream") \
    .table("de_workspace26.michaelBronze.orders_stream")

Silver


In [0]:
orders = spark.table("de_workspace26.michaelBronze.orders")
orders = orders.dropDuplicates(["order_id"])

In [0]:
from pyspark.sql.functions import col, to_date

orders = orders.withColumn("order_date", to_date(col("order_date"))) \
               .withColumn("revenue", col("quantity") * col("unit_price"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

orders = orders.withColumn(
    "cumulative_revenue",
    sum("revenue").over(window_spec)
)

In [0]:
customers = spark.table("de_workspace26.michaelBronze.customers")
products = spark.table("de_workspace26.michaelBronze.products")

orders = orders.join(customers, "customer_id", "left") \
               .join(products, "product_id", "left")

In [0]:
orders.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .saveAsTable("de_workspace26.michaelSilver.orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_workspace26.michaelSilver.customers (
  customer_id STRING,
  name STRING,
  email STRING,
  city STRING,
  loyalty_tier STRING,
  signup_date STRING,
  is_current BOOLEAN,
  effective_start_date DATE,
  effective_end_date DATE
) USING DELTA;

In [0]:
%sql
MERGE INTO de_workspace26.michaelSilver.customers AS target
USING (
  SELECT *, current_date() AS start_date
  FROM de_workspace26.michaelBronze.customers
) AS source
ON target.customer_id = source.customer_id AND target.is_current = true

WHEN MATCHED AND target.loyalty_tier <> source.loyalty_tier THEN
  UPDATE SET
    is_current = false,
    effective_end_date = current_date()

WHEN NOT MATCHED THEN
  INSERT (
    customer_id, name, email, city, loyalty_tier, signup_date,
    is_current, effective_start_date, effective_end_date
  )
  VALUES (
    source.customer_id, source.name, source.email, source.city, source.loyalty_tier, source.signup_date,
    true, source.start_date, NULL
  );

In [0]:
%sql
ALTER TABLE de_workspace26.michaelSilver.orders
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
OPTIMIZE de_workspace26.michaelSilver.orders
ZORDER BY (customer_id, order_date);

In [0]:
%sql
SHOW TABLES IN de_workspace26.michaelSilver;

In [0]:
%sql
CREATE OR REPLACE VIEW de_workspace26.michaelGold.monthly_revenue_by_region AS
SELECT
  YEAR(order_date) AS year,
  MONTH(order_date) AS month,
  region,
  SUM(revenue) AS total_revenue
FROM de_workspace26.michaelSilver.orders
GROUP BY year, month, region;

In [0]:
%sql
CREATE OR REPLACE VIEW de_workspace26.michaelGold.top_products AS
SELECT
  product_name,
  category,
  total_revenue,
  RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank
FROM (
  SELECT
    product_name,
    category,
    SUM(revenue) AS total_revenue
  FROM de_workspace26.michaelSilver.orders
  GROUP BY product_name, category
);

In [0]:
%sql
SELECT *
FROM de_workspace26.michaelGold.monthly_revenue_by_region;

In [0]:
%sql
SELECT *
FROM de_workspace26.michaelGold.top_products
WHERE rank <= 10;

In [0]:
from pyspark.sql.functions import col, to_timestamp

stream_df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/de_workspace26/michaelBronze/bronze/schema/live_orders") \
    .load(base_path)

stream_df = stream_df.withColumn("order_date", to_timestamp(col("order_date")))
stream_df = stream_df.withWatermark("order_date", "10 minutes")

query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/de_workspace26/michaelBronze/bronze/checkpoints/live_orders") \
    .table("de_workspace26.michaelGold.live_orders")

In [0]:
%sql
select count(*) from de_workspace26.michaelGold.live_orders;

In [0]:
%sql
show tables in de_workspace26.michaelBronze;